# Kickstarter Regression Preprocessing — Stage 1

This notebook prepares the raw Kickstarter CSV files for a **regression problem** where the goal is to predict the final amount raised.

It stops **before train/validation/test splitting and categorical encoding**.

Pipeline:

1. Load and merge all `Kickstarter*.csv` files
2. Check schemas
3. Remove duplicate campaign IDs
4. Keep completed campaigns only
5. Create the regression target from `usd_pledged`
6. Extract category and location information
7. Convert campaign goal to USD
8. Engineer basic text/date/video features
9. Remove target leakage
10. Remove irrelevant/redundant columns
11. Inspect categorical cardinality
12. Save the clean pre-encoding dataset


## Cell 1 — Imports


In [1]:
import os
import glob
import json
import ast
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## Cell 2 — Set the source folder

Put all files such as:

- `Kickstarter001.csv`
- `Kickstarter002.csv`
- ...
- `Kickstarter020.csv`

inside the same folder.

Change `SOURCE_DIR` if needed.


In [2]:
# Example 1: CSV files are in a local folder called "source"
SOURCE_DIR = r"E:\NSU\cse445\EDA attempt3\full raw data zip"

# Example 2: if using uploaded files in this environment:
# SOURCE_DIR = "/mnt/data"

OUTPUT_FILE = r"kickstarter_raw.csv"


## Cell 3 — Find all Kickstarter CSV files


In [ ]:
files = sorted(
    glob.glob(
        os.path.join(
            SOURCE_DIR,
            "Kickstarter_raw*.csv"
        )
    )
)

print("Number of CSV files found:", len(files))

for file in files:
    print(os.path.basename(file))

if len(files) == 0:
    raise FileNotFoundError(
        f"No Kickstarter CSV files found in: {SOURCE_DIR}"
    )


TypeError: 'function' object is not iterable

## Cell 4 — Check whether the CSV schemas are compatible

This checks whether later CSV files contain missing or extra columns compared with the first CSV.


In [4]:
reference_columns = set(
    pd.read_csv(
        files[0],
        nrows=1
    ).columns
)

schema_issues = []

for file in files:
    cols = set(
        pd.read_csv(
            file,
            nrows=1
        ).columns
    )

    missing = reference_columns - cols
    extra = cols - reference_columns

    if missing or extra:
        schema_issues.append({
            "file": os.path.basename(file),
            "missing": sorted(missing),
            "extra": sorted(extra)
        })

if schema_issues:
    print("Schema differences found:")
    for issue in schema_issues:
        print("\nFile:", issue["file"])
        print("Missing:", issue["missing"])
        print("Extra:", issue["extra"])
else:
    print("All CSV files have compatible columns.")


All CSV files have compatible columns.


## Cell 5 — Load and merge all CSV files

This works for 10, 20, 50, or more Kickstarter CSV files.


In [ ]:
dfs = []

for file in files:
    temp = pd.read_csv(
        file,
        low_memory=False
    )

    temp["source_file"] = os.path.basename(file)

    print(
        os.path.basename(file),
        "| rows =", len(temp),
        "| columns =", len(temp.columns)
    )

    dfs.append(temp)

df = pd.concat(
    dfs,
    ignore_index=True
)

print("\nMerged dataset shape:", df.shape)

# Save the merged dataset
output_file = "merged_dataset.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Saved merged dataset to: {output_file}")
print(f"Final shape: {df.shape}")

Kickstarter001.csv | rows = 3200 | columns = 43
Kickstarter002.csv | rows = 3194 | columns = 43
Kickstarter003.csv | rows = 3153 | columns = 43
Kickstarter004.csv | rows = 3190 | columns = 43
Kickstarter005.csv | rows = 3170 | columns = 43
Kickstarter006.csv | rows = 3169 | columns = 43
Kickstarter007.csv | rows = 3183 | columns = 43
Kickstarter008.csv | rows = 3191 | columns = 43
Kickstarter009.csv | rows = 3183 | columns = 43
Kickstarter010.csv | rows = 3196 | columns = 43
Kickstarter011.csv | rows = 3161 | columns = 43
Kickstarter012.csv | rows = 3167 | columns = 43
Kickstarter013.csv | rows = 3171 | columns = 43
Kickstarter014.csv | rows = 3177 | columns = 43
Kickstarter015.csv | rows = 3176 | columns = 43
Kickstarter016.csv | rows = 3193 | columns = 43
Kickstarter017.csv | rows = 3182 | columns = 43
Kickstarter018.csv | rows = 3193 | columns = 43
Kickstarter019.csv | rows = 3193 | columns = 43
Kickstarter020.csv | rows = 3155 | columns = 43

Merged dataset shape: (63597, 43)


## Cell 6 — Inspect the merged dataset


In [6]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

print("\nColumn names:")
for col in df.columns:
    print(col)


Rows: 63597
Columns: 43


,backers_count,blurb,category,converted_pledged_amount,country,country_displayable_name,created_at,creator,currency,currency_symbol,currency_trailing_code,current_currency,deadline,disable_communication,fx_rate,goal,id,is_disliked,is_in_post_campaign_pledging_phase,is_launched,is_liked,is_starrable,launched_at,location,name,percent_funded,photo,pledged,prelaunch_activated,profile,slug,source_url,spotlight,staff_pick,state,state_changed_at,static_usd_rate,urls,usd_exchange_rate,usd_pledged,usd_type,video,source_file
0,16,SLA's 2019 concept punk record is looking to m...,"{""id"":321,""name"":""Punk"",""analytics_name"":""Punk...",495.0,US,the United States,1576636334,"{""id"":1228915296,""name"":""Skyler Husebye"",""slug...",USD,$,True,USD,1582336581,False,1.000000,3300.0,1724197851,False,False,True,False,False,1577152581,"{""id"":2402292,""name"":""Fargo"",""slug"":""fargo-nd""...","""The Greatest Arsonist"" by Straight Line Arriv...",15.000000,"{""key"":""assets/027/534/169/c8f1e67e4ca56a303ce...",495.0,False,"{""id"":3873617,""project_id"":3873617,""state"":""in...",the-greatest-arsonist-by-straight-line-arrival...,https://www.kickstarter.com/discover/categorie...,False,False,failed,1582336581,1.000000,"{""web"":{""project"":""https://www.kickstarter.com...",1.000000,495.000000,domestic,NaN,Kickstarter001.csv
1,8,JOIN FANG FANG TODAY IN PRODUCING A ONE OF A K...,"{""id"":321,""name"":""Punk"",""analytics_name"":""Punk...",162.0,US,the United States,1568861333,"{""id"":1665133421,""name"":""FANG FANG"",""slug"":""fa...",USD,$,True,USD,1573275300,False,1.000000,3000.0,1922953059,False,False,True,False,False,1570358333,"{""id"":2471217,""name"":""Philadelphia"",""slug"":""ph...",FANG FANG - OH MY!,5.400000,"{""key"":""assets/026/594/103/8b54991717609a18bef...",162.0,False,"{""id"":3816191,""project_id"":3816191,""state"":""in...",fang-fang-oh-my,https://www.kickstarter.com/discover/categorie...,False,False,failed,1573275300,1.000000,"{""web"":{""project"":""https://www.kickstarter.com...",1.000000,162.000000,domestic,NaN,Kickstarter001.csv
2,189,What happens when your best friend replaces yo...,"{""id"":292,""name"":""Comedy"",""analytics_name"":""Co...",18571.0,GB,the United Kingdom,1458752846,"{""id"":1183234287,""name"":""Brendan Cleaves & Stu...",GBP,£,False,USD,1464890400,False,1.342258,10000.0,2125098884,False,False,True,False,False,1462262351,"{""id"":44418,""name"":""London"",""slug"":""london-gb""...",ROGER - Starring Game of Thrones' John Bradley...,128.760000,"{""key"":""assets/012/412/868/6978d8e055f28b3c481...",12876.0,False,"{""id"":2437094,""project_id"":2437094,""state"":""in...",roger-starring-game-of-thrones-john-bradley-an...,https://www.kickstarter.com/discover/categorie...,True,True,successful,1464890400,1.461044,"{""web"":{""project"":""https://www.kickstarter.com...",1.442325,18812.404862,domestic,"{""id"":663878,""status"":""successful"",""hls"":null,...",Kickstarter001.csv
3,27,Shooting two high-value production sketches - ...,"{""id"":292,""name"":""Comedy"",""analytics_name"":""Co...",2661.0,GB,the United Kingdom,1461600991,"{""id"":1379753904,""name"":""Mixed Doubles"",""is_re...",GBP,£,False,USD,1463847360,False,1.342258,1500.0,773841721,False,False,True,False,False,1461947129,"{""id"":44418,""name"":""London"",""slug"":""london-gb""...",Mixed Doubles Comedy Sketches,122.333333,"{""key"":""assets/012/437/955/b38319ec3cbb7438b82...",1835.0,False,"{""id"":2486783,""project_id"":2486783,""state"":""in...",mixed-doubles-comedy-sketches,https://www.kickstarter.com/discover/categorie...,True,False,successful,1463847360,1.454004,"{""web"":{""project"":""https://www.kickstarter.com...",1.450328,2668.097413,domestic,"{""id"":661089,""status"":""successful"",""hls"":null,...",Kickstarter001.csv
4,204,"A Theatrical Prequel to Hell's Rebels, the cur...","{""id"":285,""name"":""Plays"",""analytics_name"":""Pla...",7810.0,US,the United States,1433525235,"{""id"":346863196,""name"":""Cleveland High School ...",USD


Column names:
backers_count
blurb
category
converted_pledged_amount
country
country_displayable_name
created_at
creator
currency
currency_symbol
currency_trailing_code
current_currency
deadline
disable_communication
fx_rate
goal
id
is_disliked
is_in_post_campaign_pledging_phase
is_launched
is_liked
is_starrable
launched_at
location
name
percent_funded
photo
pledged
prelaunch_activated
profile
slug
source_url
spotlight
staff_pick
state
state_changed_at
static_usd_rate
urls
usd_exchange_rate
usd_pledged
usd_type
video
source_file


## Cell 7 — Remove duplicate Kickstarter campaigns

The same campaign may appear in multiple CSV snapshots.

We keep the latest record according to `state_changed_at`.


In [7]:
print("Duplicate campaign IDs before removal:")
print(df["id"].duplicated().sum())

df["_state_changed_num"] = pd.to_numeric(
    df["state_changed_at"],
    errors="coerce"
)

df = df.sort_values(
    ["id", "_state_changed_num"]
)

df = df.drop_duplicates(
    subset=["id"],
    keep="last"
)

df = df.drop(
    columns=["_state_changed_num"]
)

df = df.reset_index(drop=True)

print("\nShape after duplicate removal:")
print(df.shape)


Duplicate campaign IDs before removal:
4034

Shape after duplicate removal:
(59563, 43)


## Cell 8 — Keep only completed campaigns

For predicting **final money raised**, unfinished campaigns should not be used because their final funding is not known.

We keep:

- successful
- failed
- canceled

We remove:

- live
- submitted
- started
- other unfinished states


In [8]:
print("Campaign states before filtering:")
print(df["state"].value_counts(dropna=False))

finished_states = [
    "successful",
    "failed",
    "canceled"
]

df = df[
    df["state"].isin(finished_states)
].copy()

print("\nCampaign states after filtering:")
print(df["state"].value_counts(dropna=False))

print("\nRows remaining:", len(df))


Campaign states before filtering:
state
successful    34332
failed        18107
submitted      3094
canceled       2329
live           1058
started         642
suspended         1
Name: count, dtype: int64

Campaign states after filtering:
state
successful    34332
failed        18107
canceled       2329
Name: count, dtype: int64

Rows remaining: 54768


## Cell 9 — Create the regression target

`usd_pledged` is the final amount raised converted to USD.

We preserve it as `target_usd`.

Because funding is heavily right-skewed, we also create `log_target = log1p(target_usd)`.


In [9]:
df["target_usd"] = pd.to_numeric(
    df["usd_pledged"],
    errors="coerce"
)

df = df[
    df["target_usd"].notna()
    & np.isfinite(df["target_usd"])
    & (df["target_usd"] >= 0)
].copy()

df["log_target"] = np.log1p(
    df["target_usd"]
)

print("Target statistics:")
display(
    df["target_usd"].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


Target statistics:


count    5.476800e+04
mean     1.859493e+04
std      2.447538e+05
min      0.000000e+00
50%      2.113500e+03
75%      8.230753e+03
90%      2.496535e+04
95%      5.079587e+04
99%      2.544549e+05
max      4.676226e+07
Name: target_usd, dtype: float64

## Cell 10 — Helper function for JSON-like columns

Kickstarter stores fields such as `category` and `location` as dictionary/JSON strings.


In [10]:
def parse_dict(value):

    if isinstance(value, dict):
        return value

    if pd.isna(value):
        return {}

    try:
        return json.loads(str(value))

    except Exception:
        try:
            return ast.literal_eval(str(value))
        except Exception:
            return {}


## Cell 11 — Extract useful category fields


In [11]:
category_data = df["category"].apply(
    parse_dict
)

df["category_name"] = category_data.map(
    lambda x: x.get("name")
)

df["category_parent"] = category_data.map(
    lambda x: x.get("parent_name")
)

# Some top-level categories do not have a parent.
# Use the category itself in that case.
df["category_parent"] = (
    df["category_parent"]
    .fillna(df["category_name"])
)

display(
    df[
        [
            "category_name",
            "category_parent"
        ]
    ].head()
)


,category_name,category_parent
0,Nonfiction,Publishing
1,Narrative Film,Film & Video
2,Plays,Theater
3,3D Printing,Technology
4,Ceramics,Art


## Cell 12 — Extract useful location fields

We extract:

- location country
- location state
- location type

`location_name` is intentionally dropped later because it has very high cardinality.


In [12]:
location_data = df["location"].apply(
    parse_dict
)

df["location_name"] = location_data.map(
    lambda x: x.get("name")
)

df["location_country"] = location_data.map(
    lambda x: x.get("country")
)

df["location_state"] = location_data.map(
    lambda x: x.get("state")
)

df["location_type"] = location_data.map(
    lambda x: x.get("type")
)

display(
    df[
        [
            "location_name",
            "location_country",
            "location_state",
            "location_type"
        ]
    ].head()
)


,location_name,location_country,location_state,location_type
0,Cincinnati,US,OH,Town
1,Philadelphia,US,PA,Town
2,Leeds,GB,England,Town
3,New Delhi,IN,Delhi,Town
4,Boston,US,MA,Town


## Cell 13 — Drop `location_name`

This column can have thousands of unique place names, so we remove it at this stage.


In [13]:
df = df.drop(
    columns=["location_name"],
    errors="ignore"
)

print("location_name" in df.columns)


False


## Cell 14 — Convert campaign goal to USD

The original `goal` is in each campaign's native currency.

We use:

`goal_usd = goal × static_usd_rate`

This gives all campaigns a comparable monetary input feature.


In [14]:
df["goal"] = pd.to_numeric(
    df["goal"],
    errors="coerce"
)

df["static_usd_rate"] = pd.to_numeric(
    df["static_usd_rate"],
    errors="coerce"
)

df["goal_usd"] = (
    df["goal"]
    * df["static_usd_rate"]
)

df.loc[
    df["goal_usd"] < 0,
    "goal_usd"
] = np.nan

df["log_goal_usd"] = np.log1p(
    df["goal_usd"]
)

display(
    df[
        [
            "currency",
            "goal",
            "static_usd_rate",
            "goal_usd",
            "target_usd"
        ]
    ].head(10)
)


,currency,goal,static_usd_rate,goal_usd,target_usd
0,USD,5000.0,1.000000,5000.000000,5456.000000
1,USD,3200.0,1.000000,3200.000000,3230.000000
2,GBP,3750.0,1.298691,4870.091925,406.490339
3,USD,1200.0,1.000000,1200.000000,6940.000000
4,USD,20000.0,1.000000,20000.000000,25732.000000
5,USD,1000.0,1.000000,1000.000000,1775.000000
6,USD,9000.0,1.000000,9000.000000,19769.000000
7,USD,20000.0,1.000000,20000.000000,43142.000000
8,USD,1500.0,1.000000,1500.000000,624.000000
9,GBP,3000.0,1.571568,4714.705320,5140.600367


## Cell 15 — Clean campaign title and blurb

For standard tabular ML models, we are not using raw text yet.

Instead we create simple text-length features.


In [15]:
def clean_text(value):

    if pd.isna(value):
        return ""

    value = str(value)

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


df["name"] = df["name"].apply(
    clean_text
)

df["blurb"] = df["blurb"].apply(
    clean_text
)

df["name_char_length"] = (
    df["name"].str.len()
)

df["name_word_count"] = (
    df["name"]
    .str.split()
    .str.len()
)

df["blurb_char_length"] = (
    df["blurb"].str.len()
)

df["blurb_word_count"] = (
    df["blurb"]
    .str.split()
    .str.len()
)

display(
    df[
        [
            "name_char_length",
            "name_word_count",
            "blurb_char_length",
            "blurb_word_count"
        ]
    ].head()
)


,name_char_length,name_word_count,blurb_char_length,blurb_word_count
0,27,5,34,6
1,19,3,115,20
2,39,5,117,20
3,55,7,129,23
4,14,3,129,19


## Cell 16 — Create a video indicator


In [16]:
df["has_video"] = (
    df["video"]
    .notna()
    .astype("int8")
)

print(df["has_video"].value_counts(dropna=False))


has_video
1    36671
0    18097
Name: count, dtype: int64


## Cell 17 — Clean `prelaunch_activated`


In [17]:
def bool_to_int(value):

    if pd.isna(value):
        return 0

    if isinstance(
        value,
        (bool, np.bool_)
    ):
        return int(value)

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating
        )
    ):
        return int(value != 0)

    value = str(value).strip().lower()

    if value in {
        "true",
        "1",
        "yes",
        "y",
        "t"
    }:
        return 1

    return 0


df["prelaunch_activated"] = (
    df["prelaunch_activated"]
    .apply(bool_to_int)
    .astype("int8")
)

print(
    df["prelaunch_activated"]
    .value_counts(dropna=False)
)


prelaunch_activated
0    38918
1    15850
Name: count, dtype: int64


## Cell 18 — Convert Unix timestamps to dates


In [18]:
date_columns = [
    "created_at",
    "launched_at",
    "deadline"
]

for col in date_columns:

    df[col] = pd.to_datetime(
        pd.to_numeric(
            df[col],
            errors="coerce"
        ),
        unit="s",
        errors="coerce",
        utc=True
    )


## Cell 19 — Engineer campaign duration and launch-time features


In [19]:
df["duration_days"] = (
    (
        df["deadline"]
        - df["launched_at"]
    )
    .dt.total_seconds()
    / 86400
)

df["prelaunch_days"] = (
    (
        df["launched_at"]
        - df["created_at"]
    )
    .dt.total_seconds()
    / 86400
)

df.loc[
    df["duration_days"] < 0,
    "duration_days"
] = np.nan

df.loc[
    df["prelaunch_days"] < 0,
    "prelaunch_days"
] = np.nan

df["launch_year"] = (
    df["launched_at"].dt.year
)

df["launch_month"] = (
    df["launched_at"].dt.month
)

df["launch_day"] = (
    df["launched_at"].dt.day
)

df["launch_weekday"] = (
    df["launched_at"].dt.weekday
)

df["launch_hour"] = (
    df["launched_at"].dt.hour
)

display(
    df[
        [
            "duration_days",
            "prelaunch_days",
            "launch_year",
            "launch_month",
            "launch_day",
            "launch_weekday",
            "launch_hour"
        ]
    ].head()
)


,duration_days,prelaunch_days,launch_year,launch_month,launch_day,launch_weekday,launch_hour
0,30.000000,8.521204,2024,7,19,4,13
1,30.000000,32.095162,2013,6,3,0,16
2,28.880116,22.313322,2020,2,27,3,19
3,30.000000,43.390683,2024,9,25,2,9
4,27.610451,66.660729,2014,11,4,1,14


## Cell 20 — Remove target leakage

These fields reveal information about how the campaign actually performed.

They must **not** be included as predictors when the target is final amount raised.


In [20]:
leakage_columns = [

    # Direct money raised
    "pledged",
    "usd_pledged",
    "converted_pledged_amount",

    # Derived from funding performance
    "percent_funded",

    # Campaign performance / outcome
    "backers_count",
    "state",
    "state_changed_at",
    "spotlight",

    # Post-launch / outcome-related fields
    "is_in_post_campaign_pledging_phase",
    "staff_pick",
]

existing_leakage = [
    col for col in leakage_columns
    if col in df.columns
]

print("Leakage columns being removed:")
print(existing_leakage)

df = df.drop(
    columns=leakage_columns,
    errors="ignore"
)

# target_usd and log_target are kept separately as labels


Leakage columns being removed:
['pledged', 'usd_pledged', 'converted_pledged_amount', 'percent_funded', 'backers_count', 'state', 'state_changed_at', 'spotlight', 'is_in_post_campaign_pledging_phase', 'staff_pick']


## Cell 21 — Remove irrelevant and redundant columns

These fields are not useful as predictors in the current tabular-regression setup, or their information has already been extracted.


In [21]:
irrelevant_columns = [

    # Original JSON columns already extracted
    "category",
    "location",
    "creator",

    # Raw text already summarized with length features
    "name",
    "blurb",

    # Media / UI objects
    "video",
    "photo",
    "profile",

    # URLs and textual identifiers
    "urls",
    "source_url",
    "slug",

    # Dataset provenance
    "source_file",

    # Currency display/formatting information
    "currency_symbol",
    "currency_trailing_code",
    "country_displayable_name",

    # Exchange-rate fields not needed after goal_usd is created
    "current_currency",
    "fx_rate",
    "usd_exchange_rate",
    "usd_type",
    "static_usd_rate",

    # Original native-currency goal
    "goal",

    # Raw date columns already converted to engineered features
    "created_at",
    "launched_at",
    "deadline",

    # UI / interaction fields
    "disable_communication",
    "is_disliked",
    "is_liked",
    "is_starrable",
    "is_launched",
]

existing_irrelevant = [
    col for col in irrelevant_columns
    if col in df.columns
]

print("Irrelevant/redundant columns being removed:")
print(existing_irrelevant)

df = df.drop(
    columns=irrelevant_columns,
    errors="ignore"
)


Irrelevant/redundant columns being removed:
['category', 'location', 'creator', 'name', 'blurb', 'video', 'photo', 'profile', 'urls', 'source_url', 'slug', 'source_file', 'currency_symbol', 'currency_trailing_code', 'country_displayable_name', 'current_currency', 'fx_rate', 'usd_exchange_rate', 'usd_type', 'static_usd_rate', 'goal', 'created_at', 'launched_at', 'deadline', 'disable_communication', 'is_disliked', 'is_liked', 'is_starrable', 'is_launched']


## Cell 22 — Fill missing categorical values

We keep the categorical variables as strings for now.

Encoding will be performed in the next stage after train/validation/test splitting.


In [22]:
categorical_columns = [
    "country",
    "currency",
    "category_parent",
    "category_name",
    "location_type",
    "location_country",
    "location_state",
]

for col in categorical_columns:

    if col in df.columns:

        df[col] = (
            df[col]
            .fillna("Unknown")
            .astype(str)
        )


## Cell 23 — Select the final Stage-1 columns

`id` is retained only for row tracking and should **not** be used as an ML feature.


In [23]:
final_columns = [

    # Identifier for tracking only
    "id",

    # Numerical features
    "goal_usd",
    "log_goal_usd",
    "duration_days",
    "prelaunch_days",
    "name_char_length",
    "name_word_count",
    "blurb_char_length",
    "blurb_word_count",
    "launch_year",
    "launch_month",
    "launch_day",
    "launch_weekday",
    "launch_hour",
    "has_video",
    "prelaunch_activated",

    # Categorical features
    "country",
    "currency",
    "category_parent",
    "category_name",
    "location_type",
    "location_country",
    "location_state",

    # Regression labels
    "target_usd",
    "log_target",
]

missing_final_columns = [
    col for col in final_columns
    if col not in df.columns
]

if missing_final_columns:
    print(
        "Warning - expected columns missing:",
        missing_final_columns
    )

available_final_columns = [
    col for col in final_columns
    if col in df.columns
]

clean_df = df[
    available_final_columns
].copy()

print("Clean dataset shape:", clean_df.shape)


Clean dataset shape: (54768, 25)


## Cell 24 — Inspect missing values


In [24]:
missing_summary = (
    clean_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary[
        missing_summary > 0
    ].to_frame("missing_count")
)


,missing_count


## Cell 25 — Inspect categorical cardinality

Planned strategy for the next stage:

- low-cardinality columns: one-hot encoding
- high-cardinality columns: frequency encoding
- `location_name`: already removed


In [25]:
cardinality = []

for col in categorical_columns:

    if col in clean_df.columns:

        cardinality.append({
            "column": col,
            "unique_values": clean_df[col].nunique()
        })

cardinality_df = (
    pd.DataFrame(cardinality)
    .sort_values("unique_values")
    .reset_index(drop=True)
)

display(cardinality_df)


,column,unique_values
0,location_type,9
1,currency,15
2,category_parent,15
3,country,25
4,category_name,161
5,location_country,174
6,location_state,807


## Cell 26 — Automatically identify low- and high-cardinality categorical columns

Current rule:

- `<= 30` unique values → low cardinality
- `> 30` unique values → high cardinality

No encoding is performed yet.


In [26]:
CARDINALITY_THRESHOLD = 30

low_cardinality = []
high_cardinality = []

for col in categorical_columns:

    if col not in clean_df.columns:
        continue

    n_unique = clean_df[col].nunique()

    if n_unique <= CARDINALITY_THRESHOLD:
        low_cardinality.append(col)
    else:
        high_cardinality.append(col)

print("Low-cardinality columns:")
print(low_cardinality)

print("\nHigh-cardinality columns:")
print(high_cardinality)


Low-cardinality columns:
['country', 'currency', 'category_parent', 'location_type']

High-cardinality columns:
['category_name', 'location_country', 'location_state']


## Cell 27 — Inspect the final Stage-1 dataset


In [27]:
print("Final shape:", clean_df.shape)

print("\nFinal columns:")
for col in clean_df.columns:
    print(col)

print("\nSample rows:")
display(clean_df.head())


Final shape: (54768, 25)

Final columns:
id
goal_usd
log_goal_usd
duration_days
prelaunch_days
name_char_length
name_word_count
blurb_char_length
blurb_word_count
launch_year
launch_month
launch_day
launch_weekday
launch_hour
has_video
prelaunch_activated
country
currency
category_parent
category_name
location_type
location_country
location_state
target_usd
log_target

Sample rows:


,id,goal_usd,log_goal_usd,duration_days,prelaunch_days,name_char_length,name_word_count,blurb_char_length,blurb_word_count,launch_year,launch_month,launch_day,launch_weekday,launch_hour,has_video,prelaunch_activated,country,currency,category_parent,category_name,location_type,location_country,location_state,target_usd,log_target
0,22853,5000.000000,8.517393,30.000000,8.521204,27,5,34,6,2024,7,19,4,13,0,1,US,USD,Publishing,Nonfiction,Town,US,OH,5456.000000,8.604654
1,53154,3200.000000,8.071219,30.000000,32.095162,19,3,115,20,2013,6,3,0,16,1,0,US,USD,Film & Video,Narrative Film,Town,US,PA,3230.000000,8.080547
2,84054,4870.091925,8.491073,28.880116,22.313322,39,5,117,20,2020,2,27,3,19,1,0,GB,GBP,Theater,Plays,Town,GB,England,406.490339,6.010017
3,117526,1200.000000,7.090910,30.000000,43.390683,55,7,129,23,2024,9,25,2,9,1,1,US,USD,Technology,3D Printing,Town,IN,Delhi,6940.000000,8.845201
4,274865,20000.000000,9.903538,27.610451,66.660729,14,3,129,19,2014,11,4,1,14,1,0,US,USD,Art,Ceramics,Town,US,MA,25732.000000,10.155529


## Cell 28 — Inspect target skew


In [28]:
print("Raw target:")
display(
    clean_df["target_usd"].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\nLog-transformed target:")
display(
    clean_df["log_target"].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


Raw target:


count    5.476800e+04
mean     1.859493e+04
std      2.447538e+05
min      0.000000e+00
50%      2.113500e+03
75%      8.230753e+03
90%      2.496535e+04
95%      5.079587e+04
99%      2.544549e+05
max      4.676226e+07
Name: target_usd, dtype: float64


Log-transformed target:


count    54768.000000
mean         6.927221
std          3.051512
min          0.000000
50%          7.656574
75%          9.015754
90%         10.125284
95%         10.835590
99%         12.446883
max         17.660587
Name: log_target, dtype: float64

## Cell 29 — Save the clean pre-encoding dataset

The next notebook/stage should start from this file and perform:

1. train/validation/test split
2. one-hot encoding for low-cardinality columns
3. frequency encoding for high-cardinality columns
4. imputation/scaling if required
5. model training


In [29]:
clean_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE)
print("Shape:", clean_df.shape)


Saved: kickstarter_clean_before_encoding.csv
Shape: (54768, 25)
